# VaR and CVaR — companion notebook

Runnable companion for [`var-cvar.md`](./var-cvar.md).

1. Simulate daily P&L from Normal vs Student-$t$ with matched variance.
2. Compute VaR and CVaR three ways: parametric Normal, historical (empirical quantile), fitted Student-$t$.
3. Show that the Normal assumption under-estimates tail risk on heavy-tailed data.
4. Plot the empirical loss distribution with VaR / CVaR markers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

In [ ]:
# Match variance: a Student-t_nu scaled by sqrt((nu-2)/nu) has unit variance.
n = 10_000
sigma = 0.01  # 1% daily vol
nu = 4         # heavy tails (kurtosis is infinite below nu=4 mean-wise; nu=4 still finite var)

pnl_normal = rng.normal(0.0, sigma, size=n)
raw_t = rng.standard_t(df=nu, size=n)
pnl_t = sigma * np.sqrt((nu - 2) / nu) * raw_t  # variance = sigma^2

for name, pnl in [('Normal', pnl_normal), ('Student-t', pnl_t)]:
    print(f'{name:9s}  mean={pnl.mean():+.5f}  std={pnl.std():.5f}  '
          f'kurtosis={stats.kurtosis(pnl):+.3f}  min={pnl.min():+.4f}')

In [ ]:
alpha = 0.99

def var_cvar_parametric_normal(pnl, alpha):
    losses = -pnl
    mu_L, sig_L = losses.mean(), losses.std(ddof=1)
    z = stats.norm.ppf(alpha)
    var_ = mu_L + sig_L * z
    cvar = mu_L + sig_L * stats.norm.pdf(z) / (1 - alpha)
    return var_, cvar

def var_cvar_historical(pnl, alpha):
    losses = -pnl
    var_ = np.quantile(losses, alpha)
    cvar = losses[losses >= var_].mean()
    return var_, cvar

def var_cvar_student_t(pnl, alpha):
    losses = -pnl
    nu_hat, mu_hat, s_hat = stats.t.fit(losses)
    q = stats.t.ppf(alpha, nu_hat)
    var_ = mu_hat + s_hat * q
    # ES for standard t: (nu + q^2) / (nu - 1) * f_nu(q) / (1 - alpha)
    es_std = (nu_hat + q**2) / (nu_hat - 1) * stats.t.pdf(q, nu_hat) / (1 - alpha)
    cvar = mu_hat + s_hat * es_std
    return var_, cvar, (nu_hat, mu_hat, s_hat)

for name, pnl in [('Normal data', pnl_normal), ('Student-t data', pnl_t)]:
    print(f'\n=== {name} ===')
    v, c = var_cvar_parametric_normal(pnl, alpha)
    print(f'  Parametric Normal : VaR={v:.4f}  CVaR={c:.4f}')
    v, c = var_cvar_historical(pnl, alpha)
    print(f'  Historical        : VaR={v:.4f}  CVaR={c:.4f}')
    v, c, params = var_cvar_student_t(pnl, alpha)
    print(f'  Fitted Student-t  : VaR={v:.4f}  CVaR={c:.4f}  (nu={params[0]:.2f})')

## How wrong is Normal on heavy-tailed data?

Use the parametric-Normal VaR as if it were correct; count how often the heavy-tailed P&L exceeds it. Under a correct model the exceedance rate should be $1 - \alpha$.

In [ ]:
var_norm_on_t, _ = var_cvar_parametric_normal(pnl_t, alpha)
exceedances = (-pnl_t > var_norm_on_t).mean()
print(f'Nominal exceedance rate at alpha={alpha}: {1 - alpha:.4f}')
print(f'Observed (Normal VaR on Student-t data) : {exceedances:.4f}')
print(f'Ratio (observed / nominal)              : {exceedances / (1 - alpha):.2f}x')

# Kupiec POF likelihood-ratio test
N = len(pnl_t)
X = int((-pnl_t > var_norm_on_t).sum())
p = 1 - alpha
phat = X / N
LR = -2 * (X * np.log(p) + (N - X) * np.log(1 - p)
           - X * np.log(phat) - (N - X) * np.log(1 - phat))
pval = 1 - stats.chi2.cdf(LR, df=1)
print(f'\nKupiec LR = {LR:.2f}   p-value = {pval:.2e}  (reject Normal VaR model)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)
for ax, (name, pnl) in zip(axes, [('Normal P&L', pnl_normal), ('Student-t P&L', pnl_t)]):
    losses = -pnl
    v_hist, c_hist = var_cvar_historical(pnl, alpha)
    v_norm, c_norm = var_cvar_parametric_normal(pnl, alpha)

    ax.hist(losses, bins=80, density=True, color='lightgray', edgecolor='white')
    ax.axvline(v_hist, color='tab:blue', lw=2, label=f'historical VaR  = {v_hist:.4f}')
    ax.axvline(c_hist, color='tab:blue', lw=2, ls='--', label=f'historical CVaR = {c_hist:.4f}')
    ax.axvline(v_norm, color='tab:red', lw=2, label=f'Normal VaR     = {v_norm:.4f}')
    ax.axvline(c_norm, color='tab:red', lw=2, ls='--', label=f'Normal CVaR    = {c_norm:.4f}')
    ax.set_title(f'{name} — loss distribution at alpha={alpha}')
    ax.set_xlabel('loss'); ax.set_ylabel('density')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Zoom into the right tail to see the heavy-tail disagreement directly
fig, ax = plt.subplots(figsize=(8, 4))
grid = np.linspace(0.01, 0.08, 400)
ax.semilogy(grid, 1 - stats.norm.cdf(grid, loc=0, scale=sigma),
            'tab:red', lw=2, label='Normal tail P(L > x)')
# Empirical survival for the t data
losses_t = np.sort(-pnl_t)
surv = 1.0 - np.arange(1, len(losses_t) + 1) / len(losses_t)
ax.semilogy(losses_t, surv, 'tab:gray', lw=1, label='empirical (Student-t data)')
ax.set_xlabel('loss threshold x'); ax.set_ylabel('P(L > x) [log]')
ax.set_title('Right-tail survival: Normal vs heavy-tailed empirical')
ax.grid(True, which='both', alpha=0.3); ax.legend()
plt.show()